# Note that this project is coded for R

![Flight departing large city](flight-nyc.png)


A foremost aviation industry player with a significant presence in New York City has launched an in-depth data analysis project focused on identifying trends in flight durations in air travel. This initiative aims to delve into a wealth of data related to flight schedules and operational patterns, with the objective of optimizing flight times and enhancing the overall travel experience for passengers. As the head data analyst, you have access to rich datasets, sourced from the 'nycflights2022' collection produced by the ModernDive team. These datasets include records of flights departing from major New York City airports, including JFK (John F. Kennedy International Airport), LGA (LaGuardia Airport), and EWR (Newark Liberty International Airport), during the second half of 2022. They offer a comprehensive view of flight operations, covering various aspects such as departure and arrival times, flight paths, and airline specifics:

- `flights2022-h2.csv` contains information about each flight including 

| Variable         | Description                                              |
|------------------|----------------------------------------------------------|
| `carrier`        | Airline carrier code                                     | 
| `origin`         | Origin airport (IATA code)                               | 
| `dest`           | Destination airport (IATA code)                          | 
| `air_time`       | Duration of the flight in air, in minutes                |

- `airlines.csv` contains information about each airline:

| Variable  | Description                          |
|-----------|--------------------------------------|
| `carrier` | Airline carrier code                 |
| `name`    | Full name of the airline             |

- `airports.csv` provides details of airports:

| Variable | Description                           |
|----------|---------------------------------------|
| `faa`    | FAA code of the airport               |
| `name`   | Full name of the airport              |

In [26]:
# Import required packages
library(dplyr)
library(readr)

library(stringr)

# Load the data
flights <- read_csv("flights2022-h2.csv")
airlines <- read_csv("airlines.csv")
airports <- read_csv("airports.csv")


### Other notes
- `flights`: 218,802 rows
- `airlines`: 16 rows
- `airports`: 1,251 rows

- In "flights," most of the `air_time` values are "Invalid Number" --> cannot drop these observations.

# Objectives
- Which airline and airport pair receives the most flights from NYC and what is the average duration of that flight? Save your answer as a data frame called `frequent` with one row and a minimum of two columns: `airline_name`, `airport_name`.
- Find the airport that has the longest average flight duration (in hours) from NYC. What is the name of this airport? Save your answer as a data frame called `longest` with one row and a minimum of two columns: `airline_name`, `airport_name`.
- Which airport is the least frequented destination for flights departing from JFK? Save your answer as a character string called `least`.

# Q1
Which airline and airport pair receives the most flights from NYC and what is the average duration of that flight? Save your answer as a data frame called `frequent` with one row and a minimum of two columns: `airline_name`, `airport_name`.

- the term "receives" implies that we look at the `dest` variable in "flights"
- the term "pair" indicates to group airline & airport together
- since this data pertains to data of flights departing from NYC airports, don't need to filter further regarding location
- for average duration, use the `air_time` variable. most of these values are not a number ("Invalid Number"), so need to calculate avg without these values. fortunately, using "na.rm=TRUE" filters these values out.
- need to get `airline_name` & `airport_name` from "airlines" & "airports" respectively

In [ ]:
#Q1

q1 <- flights %>%
    #join "airlines" to get the 'airline_name'
    left_join(airlines, by="carrier") %>%
    #join "airports" to get the 'airport_name'
    left_join(airports, by=c("dest"="faa")) %>%
    #select columns of interest
    select(carrier, airline_name=name.x, dest, airport_name=name.y, air_time) %>%
    group_by(airline_name, airport_name) %>%
    #calculate # flights & avg 'air_time' per airline-airport pair
    summarize(n_flights = n(), avg_dur = mean(air_time, na.rm=TRUE)) %>%
    #ungroup the data
    ungroup() %>%
    #condense for the largest 'n_flights' value
    slice_max(n_flights, n =1)
#q1

frequent <- q1
frequent

#OUTPUT:
    #'airline_name': Delta Air Lines inc.
    #'airport_name': Hartsfield Jackson Atlanta International Airport
    #'n_flights': 5,264
    #'avg_dur': 109.2121

# Q2
Find the airport that has the longest average flight duration (in hours) from NYC. What is the name of this airport? Save your answer as a data frame called `longest` with one row and a minimum of two columns: `airline_name`, `airport_name`.

- even though it isn't clear, need to group by `airline_name` too
- again, "from" indicates we're looking at the `dest` variable
- again, need to filter out observations with `air_time` = "Invalid Number" (can use mean(na.rm=TRUE))
- need to convert `air_time` from minutes to hrs
- need to get the `airline_name` & `airport_name` from "airlines" & "airports" respectively

In [ ]:
#Q2

q2 <- flights %>%
    #join "airlines" to get 'airline_name'
    left_join(airlines, by="carrier") %>%
    #join "airports" to get 'airport_name'
    left_join(airports, by=c("dest"="faa")) %>%
    #convert 'air_time' from minutes to hours
    mutate(air_time_hr = air_time/60) %>%
    #condense data to variables of interest
    select(carrier, airline_name=name.x, dest, airport_name=name.y, air_time_hr) %>%
    group_by(airline_name, airport_name) %>%
    #get avg 'air_time' per airline-airport pair
    summarize(avg_dur_hr = mean(air_time_hr)) %>%
    #ungroup the data
    ungroup() %>%
    #get the row with the largest avg_dur_hr
    slice_max(avg_dur_hr, n=1)
#q2

longest <- q2

#OUTPUT:
    #'airline_name': Delta Air Lines inc.
    #'airport_name': Daniel K Inouye International Airport
    #'avg_dur_hr': 10.7167

# Q3
Which airport is the least frequented destination for flights departing from JFK? Save your answer as a character string called `least`.

- can specify 'origin' == "JFK"
- need to join "airports" to get 'airport_name'

In [ ]:
#Q3

q3 <- flights %>%
    #filter for flights from JFK
    filter(origin=="JFK") %>%
    #group by the destination (airport)
    group_by(dest) %>%
    #count # flights
    summarize(n_flights = n()) %>%
    #ungroup the data
    ungroup() %>%
    #join "airports" to get the 'airport_name'
    inner_join(airports, by=c("dest"="faa")) %>%
    #select variables of interest
    select(dest:n_flights, airport_name=name) %>%
    #get the row with the fewest # flights
    slice_min(n_flights, n=1)
#q3
least <- q3$airport_name
least

#OUTPUT:
    #is a character string -- GOOD
    #"Eagle County Regional Airport"